# 05 — Unified Evaluation (YOLOv8 + Faster R-CNN)

Evaluate both models on the same test set using identical metrics:
- mAP @0.5 and @0.5:0.95
- Per-class AP
- Precision, Recall, F1
- Inference Speed (FPS)
- Model Size & Parameters
- Confusion Matrices
- PR Curves
- Qualitative Sample Detections

In [ ]:
import shutil
import os

source_dir = '/kaggle/input/datasets/bddk100k-object-detection/Object-Detection'
destination_dir = '/kaggle/working/bdd100k_project'

if os.path.exists(source_dir):
    shutil.copytree(source_dir, destination_dir, dirs_exist_ok=True)
    print("Files copied to /kaggle/working/src and ready to use!")
else:
    print("Source directory not found. Check your paths!")

In [ ]:
!pip install ultralytics --no-deps

In [ ]:
import sys
sys.path.insert(0, "/kaggle/working")

import os
import json
import torch
import numpy as np
import pandas as pd
import cv2
import random

from src.utils import SEED, CLASS_MAP, FRCNN_CLASS_MAP, NUM_CLASSES, CLASS_NAMES, seed_everything
from src.fasterrcnn_dataset import BDD100KDataset
from src.fasterrcnn_utils import build_fasterrcnn, collate_fn
from src.evaluate import (
    predict_yolov8, predict_fasterrcnn,
    compute_map, compute_precision_recall_f1,
    measure_fps, get_model_size_mb, count_params,
    plot_pr_curve, build_confusion_matrix,
    draw_boxes, plot_sample_detections, plot_per_class_ap,
)

seed_everything(SEED)

## 1. Load Models

In [ ]:
from ultralytics import YOLO

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

yolo_model = YOLO("/kaggle/working/yolov8m_bdd100k_best.pt")

frcnn_model = build_fasterrcnn(num_classes=NUM_CLASSES)
frcnn_model.load_state_dict(torch.load("/kaggle/working/outputs/fasterrcnn_bdd100k_best.pth", map_location=device))
frcnn_model.to(device)
frcnn_model.eval()

print("Both models loaded successfully")

## 2. Load Test Dataset

In [ ]:
DATASET_ROOT = "/kaggle/input/bdd100k-yolo-subset-v1"

test_dataset = BDD100KDataset(
    image_dir=f"{DATASET_ROOT}/images/test",
    annotation_file=f"{DATASET_ROOT}/annotations/test.json",
    class_map=FRCNN_CLASS_MAP,
)

print(f"Test samples: {len(test_dataset)}")

## 3. Run Inference on Test Set

In [ ]:
from tqdm import tqdm
from torch.utils.data import DataLoader

test_loader = DataLoader(test_dataset, batch_size=1, shuffle=False, collate_fn=collate_fn)

yolo_predictions = []
frcnn_predictions = []
all_targets = []

for images, targets in tqdm(test_loader, desc="Running inference"):
    img_tensor = images[0]
    target = targets[0]

    image_id = target["image_id"].item()
    item = test_dataset.annotations[image_id]
    img_path = os.path.join(test_dataset.image_dir, item["name"])

    yolo_preds = predict_yolov8(yolo_model, img_path)
    frcnn_preds = predict_fasterrcnn(frcnn_model, img_tensor, device)

    yolo_boxes = torch.tensor([p["box"] for p in yolo_preds], dtype=torch.float32) if yolo_preds else torch.zeros((0, 4))
    yolo_scores = torch.tensor([p["score"] for p in yolo_preds]) if yolo_preds else torch.zeros(0)
    yolo_labels = torch.tensor([p["label"] for p in yolo_preds], dtype=torch.int64) if yolo_preds else torch.zeros(0, dtype=torch.int64)

    frcnn_boxes = torch.tensor([p["box"] for p in frcnn_preds], dtype=torch.float32) if frcnn_preds else torch.zeros((0, 4))
    frcnn_scores = torch.tensor([p["score"] for p in frcnn_preds]) if frcnn_preds else torch.zeros(0)
    frcnn_labels = torch.tensor([p["label"] for p in frcnn_preds], dtype=torch.int64) if frcnn_preds else torch.zeros(0, dtype=torch.int64)

    yolo_predictions.append({"boxes": yolo_boxes, "scores": yolo_scores, "labels": yolo_labels})
    frcnn_predictions.append({"boxes": frcnn_boxes, "scores": frcnn_scores, "labels": frcnn_labels})
    all_targets.append({"boxes": target["boxes"], "labels": target["labels"]})

print(f"Inference complete: {len(yolo_predictions)} images")

## 4. mAP Computation

In [ ]:
print("Computing mAP for YOLOv8...")
yolo_map = compute_map(yolo_predictions, all_targets)

print("Computing mAP for Faster R-CNN...")
frcnn_map = compute_map(frcnn_predictions, all_targets)

print("\n--- YOLOv8 mAP ---")
print(f"mAP@0.5:0.95: {yolo_map['map'].item():.4f}")
print(f"mAP@0.5:     {yolo_map['map_50'].item():.4f}")
print(f"mAP@0.75:    {yolo_map['map_75'].item():.4f}")
print(f"mAP (small): {yolo_map['map_small'].item():.4f}")
print(f"mAP (medium):{yolo_map['map_medium'].item():.4f}")
print(f"mAP (large): {yolo_map['map_large'].item():.4f}")

print("\n--- Faster R-CNN mAP ---")
print(f"mAP@0.5:0.95: {frcnn_map['map'].item():.4f}")
print(f"mAP@0.5:     {frcnn_map['map_50'].item():.4f}")
print(f"mAP@0.75:    {frcnn_map['map_75'].item():.4f}")
print(f"mAP (small): {frcnn_map['map_small'].item():.4f}")
print(f"mAP (medium):{frcnn_map['map_medium'].item():.4f}")
print(f"mAP (large): {frcnn_map['map_large'].item():.4f}")

## 5. Per-Class AP

In [ ]:
CLASS_NAMES_LIST = [CLASS_NAMES[i] for i in range(len(CLASS_MAP))]

yolo_per_class = yolo_map["map_per_class"].numpy()
frcnn_per_class = frcnn_map["map_per_class"].numpy()

per_class_df = pd.DataFrame({
    "Class": CLASS_NAMES_LIST,
    "YOLOv8 AP": yolo_per_class,
    "Faster R-CNN AP": frcnn_per_class,
})
print(per_class_df.to_string(index=False))

plot_per_class_ap(
    yolo_per_class, frcnn_per_class, CLASS_NAMES_LIST,
    save_path="/kaggle/working/results/plots/per_class_ap_comparison.png"
)

## 6. Precision, Recall, F1

In [ ]:
yolo_precision, yolo_recall, yolo_f1 = compute_precision_recall_f1(yolo_predictions, all_targets)
frcnn_precision, frcnn_recall, frcnn_f1 = compute_precision_recall_f1(frcnn_predictions, all_targets)

print(f"YOLOv8      — Precision: {yolo_precision:.4f} | Recall: {yolo_recall:.4f} | F1: {yolo_f1:.4f}")
print(f"Faster R-CNN — Precision: {frcnn_precision:.4f} | Recall: {frcnn_recall:.4f} | F1: {frcnn_f1:.4f}")

## 7. Inference Speed (FPS)

In [ ]:
FPS_SAMPLE_SIZE = 200
random.seed(SEED)
fps_indices = random.sample(range(len(test_dataset)), min(FPS_SAMPLE_SIZE, len(test_dataset)))
fps_items = [test_dataset[i] for i in fps_indices]
fps_image_tensors = [item[0] for item in fps_items]
fps_image_paths = []
for idx in fps_indices:
    item = test_dataset.annotations[idx]
    fps_image_paths.append(os.path.join(test_dataset.image_dir, item["name"]))

yolo_fps, yolo_ms = measure_fps(
    lambda img: predict_yolov8(yolo_model, img),
    fps_image_paths,
    n_warmup=10
)

frcnn_fps, frcnn_ms = measure_fps(
    lambda img: predict_fasterrcnn(frcnn_model, img, device),
    fps_image_tensors,
    n_warmup=10
)

print(f"YOLOv8      — FPS: {yolo_fps:.1f} | Latency: {yolo_ms:.1f} ms/image")
print(f"Faster R-CNN — FPS: {frcnn_fps:.1f} | Latency: {frcnn_ms:.1f} ms/image")

## 8. Model Size & Parameters

In [ ]:
yolo_size = get_model_size_mb("/kaggle/working/yolov8m_bdd100k_best.pt")
frcnn_size = get_model_size_mb("/kaggle/working/outputs/fasterrcnn_bdd100k_best.pth")

yolo_params = count_all_params(yolo_model.model)
frcnn_params = count_all_params(frcnn_model)

print(f"YOLOv8       — Size: {yolo_size:.1f} MB | Parameters: {yolo_params:,}")
print(f"Faster R-CNN — Size: {frcnn_size:.1f} MB | Parameters: {frcnn_params:,}")

## 9. Confusion Matrices

In [ ]:
build_confusion_matrix(
    yolo_predictions, all_targets, CLASS_NAMES_LIST,
    iou_threshold=0.5, model_name="YOLOv8",
    save_path="/kaggle/working/results/plots/confusion_matrix_yolov8.png"
)

build_confusion_matrix(
    frcnn_predictions, all_targets, CLASS_NAMES_LIST,
    iou_threshold=0.5, model_name="Faster R-CNN",
    save_path="/kaggle/working/results/plots/confusion_matrix_fasterrcnn.png"
)

## 10. PR Curves

In [ ]:
if "precision" in yolo_map and "recall" in yolo_map:
    plot_pr_curve(
        yolo_map["precision"].mean(dim=0).numpy(),
        yolo_map["recall"].numpy(),
        "YOLOv8",
        "/kaggle/working/results/plots/pr_curve_yolov8.png"
    )

if "precision" in frcnn_map and "recall" in frcnn_map:
    plot_pr_curve(
        frcnn_map["precision"].mean(dim=0).numpy(),
        frcnn_map["recall"].numpy(),
        "Faster R-CNN",
        "/kaggle/working/results/plots/pr_curve_fasterrcnn.png"
    )

## 11. Qualitative Sample Detections

In [ ]:
random.seed(SEED)
N_SAMPLES = 6
sample_indices = random.sample(range(len(test_dataset)), N_SAMPLES)

test_items = []
for idx in sample_indices:
    img_tensor, target = test_dataset[idx]
    item = test_dataset.annotations[idx]
    img_path = os.path.join(test_dataset.image_dir, item["name"])

    gt_boxes = target["boxes"].numpy().tolist()
    gt_labels = target["labels"].numpy().tolist()

    test_items.append({
        "image_path": img_path,
        "gt_boxes": gt_boxes,
        "gt_labels": gt_labels,
    })

plot_sample_detections(
    test_items, yolo_model, frcnn_model, CLASS_NAMES_LIST, device,
    n=N_SAMPLES,
    save_path="/kaggle/working/results/plots/qualitative_samples.png"
)

## 12. Comparison Table

In [ ]:
with open("/kaggle/working/outputs/fasterrcnn_loss_history.json") as f:
    frcnn_history = json.load(f)
frcnn_train_hours = frcnn_history.get("training_time_hours", 0)

comparison = pd.DataFrame({
    "Metric": [
        "mAP@0.5", "mAP@0.5:0.95", "mAP@0.75",
        "mAP (small objects)", "mAP (medium objects)", "mAP (large objects)",
        "Precision", "Recall", "F1 Score",
        "Inference Speed (FPS)", "Latency (ms/image)",
        "Model Size (MB)", "Parameters (M)", "Training Time (hrs)",
    ],
    "YOLOv8m": [
        f"{yolo_map['map_50'].item():.4f}",
        f"{yolo_map['map'].item():.4f}",
        f"{yolo_map['map_75'].item():.4f}",
        f"{yolo_map['map_small'].item():.4f}",
        f"{yolo_map['map_medium'].item():.4f}",
        f"{yolo_map['map_large'].item():.4f}",
        f"{yolo_precision:.4f}", f"{yolo_recall:.4f}", f"{yolo_f1:.4f}",
        f"{yolo_fps:.1f}", f"{yolo_ms:.1f}",
        f"{yolo_size:.1f}", f"{yolo_params / 1e6:.1f}",
        "—",
    ],
    "Faster R-CNN": [
        f"{frcnn_map['map_50'].item():.4f}",
        f"{frcnn_map['map'].item():.4f}",
        f"{frcnn_map['map_75'].item():.4f}",
        f"{frcnn_map['map_small'].item():.4f}",
        f"{frcnn_map['map_medium'].item():.4f}",
        f"{frcnn_map['map_large'].item():.4f}",
        f"{frcnn_precision:.4f}", f"{frcnn_recall:.4f}", f"{frcnn_f1:.4f}",
        f"{frcnn_fps:.1f}", f"{frcnn_ms:.1f}",
        f"{frcnn_size:.1f}", f"{frcnn_params / 1e6:.1f}",
        f"{frcnn_train_hours:.2f}",
    ],
})

print(comparison.to_string(index=False))

comparison.to_csv("/kaggle/working/results/metrics/comparison_table.csv", index=False)
print("\nComparison table saved to results/metrics/comparison_table.csv")